# Organizado

**Status:** Versão de referência · **Área:** ideb-regiao-norte

Apesar do nome Organizado, mistura nomes de colunas derivados em df com colunas criadas em df_norte.

Este arquivo está identificado como **Versão de referência**. A organização e a revisão estática não equivalem à validação completa da análise. Consulte as pendências antes de usar seus resultados.

## Como retomar

1. Leia o README desta pasta e suas pendências.
2. Prepare os dados conforme data/README.md quando existir.
3. Reinicie o ambiente e execute em ordem; registre a verificação no README.

[Catálogo](https://github.com/Inocentrax/portfolio-dados/blob/main/CATALOGO.md) · [Plano de continuidade](https://github.com/Inocentrax/portfolio-dados/blob/main/PROXIMOS_PASSOS.md)

## Origem e créditos

Importado de um notebook pessoal do Drive em 15/11/2026. Créditos de cursos, livros e coautores no material original foram preservados. Outputs e metadados pessoais do Colab foram retirados.

---

## Desenvolvimento


Subir arquivo e Descompactar arquivo

In [ ]:
import zipfile
def descompactar_arquivos(nome_arquivo_zip, destino):
  """
  Descompacta um arquivo zip.

  Args:
    divulgacao_anos_finais_municipios_2023(1).zip: O nome do arquivo zip a ser descompactado.
    destino: O diretório onde os arquivos serão extraídos.
  """
  try:
    with zipfile.ZipFile(nome_arquivo_zip, 'r') as zip_ref:
      zip_ref.extractall(destino)
    print(f"Arquivo {nome_arquivo_zip} descompactado com sucesso para {destino}")
  except FileNotFoundError:
    print(f"Arquivo {nome_arquivo_zip} não encontrado.")
  except Exception as e:
    print(f"Erro ao descompactar o arquivo: {e}")

In [ ]:
zip_path = "/content/divulgacao_anos_finais_municipios_2023.zip"
extract_path = "./files_cand"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

Efetuar leitura

In [ ]:
import pandas as pd

df = pd.read_excel('/content/files_cand/divulgacao_anos_finais_municipios_2023/divulgacao_anos_finais_municipios_2023.xlsx', skiprows=9)
df.head()

criação de filtro da região norte

In [ ]:
regiao_norte = ['AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO']
df_norte = df[df['SG_UF'].isin(regiao_norte)].copy()

In [ ]:
# 🔢 Seleção e renomeação de colunas
colunas_norte = [
    "CO_MUNICIPIO", "REDE", "SG_UF",
    "VL_OBSERVADO_2005", "VL_OBSERVADO_2023",
    "VL_APROVACAO_2005_4", "VL_APROVACAO_2023_4",
    "VL_NOTA_PORTUGUES_2005", "VL_NOTA_PORTUGUES_2023",
    "VL_NOTA_MATEMATICA_2005", "VL_NOTA_MATEMATICA_2023"
]

df_norte = df_norte[colunas_norte].copy()

df_norte.rename(columns={
    "VL_OBSERVADO_2005": "IDEB_2005_4EF",
    "VL_OBSERVADO_2023": "IDEB_2023_4EF",
    "VL_APROVACAO_2005_4": "Aprov_2005_4EF",
    "VL_APROVACAO_2023_4": "Aprov_2023_4EF",
    "VL_NOTA_PORTUGUES_2005": "SAEB_PORT_2005",
    "VL_NOTA_PORTUGUES_2023": "SAEB_PORT_2023",
    "VL_NOTA_MATEMATICA_2005": "SAEB_MAT_2005",
    "VL_NOTA_MATEMATICA_2023": "SAEB_MAT_2023"
}, inplace=True)


In [ ]:
# 🧼 Conversão correta para tipo numérico
colunas_para_converter = [
    'IDEB_2023_4EF', 'IDEB_2005_4EF',
    'Aprov_2023_4EF', 'Aprov_2005_4EF',
    'SAEB_PORT_2023', 'SAEB_PORT_2005',
    'SAEB_MAT_2023', 'SAEB_MAT_2005'
]
df_norte[colunas_para_converter] = df_norte[colunas_para_converter].apply(pd.to_numeric, errors='coerce')
df_norte[colunas_para_converter] = df_norte[colunas_para_converter].fillna(df_norte[colunas_para_converter].mean())

# Preencher REDE e SG_UF se tiver valores nulos
df_norte['REDE'] = df_norte['REDE'].fillna('Não Informado')
df_norte['SG_UF'] = df_norte['SG_UF'].fillna('Não Informado')


In [ ]:
# 🧮 Cálculo de deltas
df_norte["Δ_IDEB"] = df_norte["IDEB_2023_4EF"] - df_norte["IDEB_2005_4EF"]
df_norte["Δ_Aprov"] = df_norte["Aprov_2023_4EF"] - df_norte["Aprov_2005_4EF"]
df_norte["Δ_Port"] = df_norte["SAEB_PORT_2023"] - df_norte["SAEB_PORT_2005"]
df_norte["Δ_Mat"] = df_norte["SAEB_MAT_2023"] - df_norte["SAEB_MAT_2005"]


In [ ]:
# 📊 Criação das variáveis categóricas para Apriori
df_norte['categoria_IDEB_2023'] = pd.cut(
    df_norte['IDEB_2023_4EF'],
    bins=[0, 4, 5, 6, 7, 10],
    labels=["muito_baixo", "baixo", "medio", "bom", "excelente"]
)

df_norte['evolucao_IDEB'] = df_norte['Δ_IDEB'].apply(
    lambda x: "melhorou" if x > 0.2 else ("piorou" if x < -0.2 else "estavel")
)


In [ ]:
# 📊 Preparação para clusterização
colunas_essenciais = [
    'IDEB_2005', 'IDEB_2023', 'VARIACAO_IDEB',
    'MEDIA_APROVACAO_2005', 'MEDIA_APROVACAO_2023', 'VARIACAO_APROVACAO',
    'SAEB_PORT_2005', 'SAEB_PORT_2023', 'SAEB_MAT_2005', 'SAEB_MAT_2023'
]
df = df.dropna(subset=colunas_essenciais)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import networkx as nx


In [ ]:
# 📊 Preparação para clusterização
variaveis_modelo = ["Δ_IDEB", "Δ_Aprov"]
df_modelo_norte = df_norte[variaveis_modelo].dropna().drop_duplicates()
scaler = StandardScaler()
dados_normalizados_norte = scaler.fit_transform(df_modelo_norte)


In [ ]:
# 📈 KMeans com k=4
kmeans_4 = KMeans(n_clusters=4, random_state=42, n_init=10)
df_modelo_norte["Cluster_4"] = kmeans_4.fit_predict(dados_normalizados_norte)

# 🔍 PCA para visualização
pca = PCA(n_components=2)
pca_result_4 = pca.fit_transform(dados_normalizados_norte)

plt.figure(figsize=(10, 6))
for cluster in range(4):
    plt.scatter(
        pca_result_4[df_modelo_norte["Cluster_4"] == cluster, 0],
        pca_result_4[df_modelo_norte["Cluster_4"] == cluster, 1],
        label=f"Cluster {cluster}", alpha=0.6
    )
plt.title("Visualização dos Clusters (k = 4) via PCA")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(df_norte["SAEB_PORT_2023"], bins=20, alpha=0.7, label='Português', color='#66c2a5')
plt.hist(df_norte["SAEB_MAT_2023"], bins=20, alpha=0.7, label='Matemática', color='#fc8d62')
plt.title("Distribuição das Notas - SAEB 2023", fontsize=14)
plt.xlabel("Nota")
plt.ylabel("Frequência")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_norte[["SAEB_PORT_2023", "SAEB_MAT_2023"]])
plt.title("Boxplot das Notas de Português e Matemática (2023)", fontsize=13)
plt.ylabel("Nota")
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
df_norte.groupby("REDE")[["SAEB_PORT_2023", "SAEB_MAT_2023"]].mean().round(2)


In [ ]:
# 📊 Radar Chart com personas
df_radar = df_norte.copy()
df_radar["Cluster"] = df_modelo_norte["Cluster_4"]
variaveis = ["IDEB_2023_4EF", "Δ_IDEB", "Aprov_2023_4EF", "Δ_Aprov", "SAEB_PORT_2023", "SAEB_MAT_2023"]
labels_bonitos = ["IDEB_2023", "VARIACAO_IDEB", "MEDIA_APROVACAO_2023", "VARIACAO_APROVACAO", "SAEB_PORT_2023", "SAEB_MAT_2023"]
df_radar = df_radar.dropna(subset=variaveis)
df_grouped = df_radar.groupby("Cluster")[variaveis].mean()
df_grouped_norm = (df_grouped - df_grouped.min()) / (df_grouped.max() - df_grouped.min())

num_clusters = df_grouped_norm.shape[0]
num_vars = len(variaveis)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]
cores = ['#6cc2b5', '#ffa07a', '#7082bd', '#f57257']
nomes_personas = ['Persona A', 'Persona B', 'Persona C', 'Persona D']
fig, axs = plt.subplots(1, num_clusters, figsize=(18, 5), subplot_kw=dict(polar=True))

for i in range(num_clusters):
    valores = df_grouped_norm.iloc[i].tolist()
    valores += valores[:1]
    axs[i].plot(angles, valores, color=cores[i], linewidth=2)
    axs[i].fill(angles, valores, color=cores[i], alpha=0.25)
    axs[i].set_title(
        f"{nomes_personas[i]} - {df_radar[df_radar['Cluster'] == i].shape[0]} registros",
        size=12, color=cores[i]
    )
    axs[i].set_xticks(angles[:-1])
    axs[i].set_xticklabels(labels_bonitos, fontsize=9)
    axs[i].set_yticklabels([])

plt.suptitle("📈 Perfil dos Clusters - Radar Chart", fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

In [ ]:
# 📊 Criação das variáveis categóricas
df_norte['categoria_IDEB_2023'] = pd.cut(
    df_norte['IDEB_2023_4EF'],
    bins=[0, 4, 5, 6, 7, 10],
    labels=["muito_baixo", "baixo", "medio", "bom", "excelente"]
)

df_norte['evolucao_IDEB'] = df_norte['Δ_IDEB'].apply(
    lambda x: "melhorou" if x > 0.2 else ("piorou" if x < -0.2 else "estavel")
)


In [ ]:
# 🔁 Preparação final para o algoritmo Apriori
# Garantir que colunas categóricas estejam preenchidas corretamente
df_norte['categoria_IDEB_2023'] = df_norte['categoria_IDEB_2023'].cat.add_categories('desconhecido').fillna('desconhecido')
df_norte['evolucao_IDEB'] = df_norte['evolucao_IDEB'].fillna('estavel')  # padrão seguro
df_norte['REDE'] = df_norte['REDE'].fillna('Não Informado')
df_norte['SG_UF'] = df_norte['SG_UF'].fillna('Não Informado')

# Preparar o df_apriori sem perder tudo no dropna
df_apriori = pd.get_dummies(df_norte[["categoria_IDEB_2023", "evolucao_IDEB", "REDE", "SG_UF"]])
df_apriori = df_apriori.astype(bool)

df_apriori = pd.get_dummies(df_apriori)
df_apriori = df_apriori.astype(bool)
print("✅ df_apriori pronto:", df_apriori.shape)


In [ ]:
print("df_apriori:", df_apriori.shape)
display(df_apriori.head())


In [ ]:
# 📦 Biblioteca Apriori
from mlxtend.frequent_patterns import apriori, association_rules

if df_apriori.shape[0] == 0:
    print("❌ df_apriori está vazio. Verifique os dados.")
else:
    # Executa o Apriori com suporte mínimo de 1%
    itens_frequentes = apriori(df_apriori, min_support=0.01, use_colnames=True)

    if itens_frequentes.empty:
        print("⚠️ Nenhum item frequente encontrado. Tente reduzir o min_support.")
    else:
        # Gera regras de associação
        regras = association_rules(itens_frequentes, metric="lift", min_threshold=1.0)

        # Filtro de confiança e lift
        regras_filtradas = regras[
            (regras['lift'] >= 1.2) & (regras['confidence'] >= 0.6)
        ].sort_values(by='lift', ascending=False)

        display(regras_filtradas)


In [ ]:
# 📊 Visualização estilizada da tabela de regras
def destacar_regras(val):
    if isinstance(val, float) and val >= 1.2:
        return 'background-color: #d0f0c0'  # verde claro
    return ''

regras_filtradas[['antecedents', 'consequents', 'support', 'confidence', 'lift']].style\
    .applymap(destacar_regras, subset=['lift'])\
    .format({
        'support': '{:.2%}',
        'confidence': '{:.2%}',
        'lift': '{:.2f}'
    })\
    .set_caption("📋 Regras de Associação Filtradas (lift ≥ 1.2, confiança ≥ 60%)")


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

if 'regras_filtradas' in locals() and not regras_filtradas.empty:
    G = nx.DiGraph()

    for _, row in regras_filtradas.iterrows():
        antecedente = ', '.join(list(row['antecedents']))
        consequente = ', '.join(list(row['consequents']))
        G.add_edge(antecedente, consequente, weight=row['lift'])

    plt.figure(figsize=(14, 9))
    pos = nx.spring_layout(G, k=0.7)

    # Cores e pesos
    weights = [G[u][v]['weight'] for u, v in G.edges()]
    nodes = nx.draw_networkx_nodes(G, pos, node_size=3000, node_color='#cce5ff')
    edges = nx.draw_networkx_edges(G, pos, width=weights, edge_color='#1f78b4', alpha=0.7)
    labels = nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold')

    plt.title("🔗 Grafo de Regras de Associação", fontsize=16)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("⛔ Nenhuma regra disponível para gerar grafo.")


In [ ]:
top10 = regras_filtradas.sort_values(by='lift', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(
    [f"{', '.join(x)} → {', '.join(y)}" for x, y in zip(top10['antecedents'], top10['consequents'])],
    top10['lift'],
    color='#3399ff'
)
plt.xlabel("Lift")
plt.title("🚀 Top 10 Regras com Maior Lift")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Continuidade

Resolver df.dropna(subset=colunas_essenciais) com nomes não construídos naquele fluxo; aproveitar gráficos úteis e arquivar a cópia após consolidação.

Registre decisões e resultados verificados no README desta pasta. Evite criar “Untitled” ou “Cópia de” para continuar este trabalho; use o mesmo arquivo e o histórico do Git.